In [73]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [29]:
TECHNIQUE_MAPPING = {
    "Attack_on_Reputation": ['Name_Calling-Labeling', 'Guilt_by_Association', 'Doubt', 'Appeal_to_Hypocrisy', 'Questioning_the_Reputation'],
    "Justification": ['Flag_Waving', 'Appeal_to_Authority', 'Appeal_to_Popularity', 'Appeal_to_Fear-Prejudice', 'Appeal_to_Values'],
    "Distraction": ['Straw_Man', 'Red_Herring', 'Whataboutism', 'Appeal_to_Pity'],
    "Simplification": ['Causal_Oversimplification', 'False_Dilemma-No_Choice', 'False_Equivalence', 'Consequential_Oversimplification'],
    "Call": ['Slogans', 'Conversation_Killer', 'Appeal_to_Time'],
    "Manipulative_Wording": ['Loaded_Language', 'Obfuscation-Vagueness-Confusion', 'Exaggeration-Minimisation', 'Repetition']
}
TECHNIQUE_MAPPING.keys()

dict_keys(['Attack_on_Reputation', 'Justification', 'Distraction', 'Simplification', 'Call', 'Manipulative_Wording'])

In [32]:
def get_high_level_persuasion_group(list_with_values):

    # Find all high-level persuasion groups matching any value in the list
    high_level_persuasion_groups = [
        group for group, keywords in TECHNIQUE_MAPPING.items()
        if any(element in keywords for element in list_with_values)
    ]

    return high_level_persuasion_groups

In [33]:
def read_transcript_to_dataframe(filepath, column):
    # Read the tab-separated file
    df = pd.read_csv(
        filepath,
        sep='\t',
        header=None,
        names=['filename', 'start', 'end', column],
        encoding='utf-8',
        quoting=3  # avoids issues if text contains quotes
    )

    # Convert start and end columns to integers
    df['start'] = pd.to_numeric(df['start'], errors='coerce').astype('Int64')
    df['end'] = pd.to_numeric(df['end'], errors='coerce').astype('Int64')

    # Drop any rows that didn't parse correctly
    df = df.dropna(subset=['start', 'end', column]).reset_index(drop=True)

    return df

In [34]:
def read_persuasion_file(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue  # skip malformed lines

            filename = parts[0]
            try:
                start = int(parts[1])
                end = int(parts[2])
            except ValueError:
                continue  # skip if start/end are not valid integers

            # Remaining columns (if any) are persuasion strategies
            strategies = parts[3:] if len(parts) > 3 else []

            rows.append({
                'filename': filename,
                'start': start,
                'end': end,
                'techniques': strategies
            })

    # Convert to DataFrame
    df = pd.DataFrame(rows, columns=['filename', 'start', 'end', 'techniques'])
    return df

In [35]:
def encode_persuasion_strategies(df, column, strategies):
    """
    Encode persuasion strategies into one-hot encoded binary columns.
    
    Args:
    df (pd.DataFrame): DataFrame containing the column with list of strategies.
    column (str): Column name containing the list of strategies.
    strategies (list): List of all possible persuasion strategies.
    
    Returns:
    pd.DataFrame: DataFrame with binary encoded columns for each strategy.
    """
    # Iterate through each strategy and create binary columns
    for strategy in strategies:
        df[strategy] = df[column].apply(lambda x: True if strategy in x else False)
    
    return df

# Polish Dataset

In [36]:
# change to your actual file path
file_path = "../data/PL/sentences.txt" # replace with your file path
df_content = read_transcript_to_dataframe(file_path, "content")
df_content.head()

,filename,start,end,content
0,pl_eu_12_06_2024_n02.txt,0,53,Wicemarszałek Włodzimierz Czarzasty: Dziękuję ...
1,pl_eu_12_06_2024_n02.txt,54,143,"Witam uczniów ze Szkoły Podstawowej nr 4, szko..."
2,pl_eu_12_06_2024_n02.txt,144,256,"Witam państwa. (Oklaski) Proszę państwa, przec..."
3,pl_eu_12_06_2024_n02.txt,257,296,Ustalam czas zadania pytania na minutę.
4,pl_eu_12_06_2024_n02.txt,297,380,Zapraszam panią poseł Annę Gembicką z Klubu Pa...


In [37]:
file_path = "../data/PL/subtask-2-annotations-full.txt" # replace with your file path
df_labels = read_persuasion_file(file_path)

In [38]:
df_labels.head()

,filename,start,end,techniques
0,pl_eu_12_06_2024_n02.txt,0,53,[]
1,pl_eu_12_06_2024_n02.txt,54,143,[]
2,pl_eu_12_06_2024_n02.txt,144,256,[]
3,pl_eu_12_06_2024_n02.txt,257,296,[]
4,pl_eu_12_06_2024_n02.txt,297,380,[]


In [39]:
df=df_content.merge(df_labels, on=["filename", "start", "end"])

In [40]:
df.head()

,filename,start,end,content,techniques
0,pl_eu_12_06_2024_n02.txt,0,53,Wicemarszałek Włodzimierz Czarzasty: Dziękuję ...,[]
1,pl_eu_12_06_2024_n02.txt,54,143,"Witam uczniów ze Szkoły Podstawowej nr 4, szko...",[]
2,pl_eu_12_06_2024_n02.txt,144,256,"Witam państwa. (Oklaski) Proszę państwa, przec...",[]
3,pl_eu_12_06_2024_n02.txt,257,296,Ustalam czas zadania pytania na minutę.,[]
4,pl_eu_12_06_2024_n02.txt,297,380,Zapraszam panią poseł Annę Gembicką z Klubu Pa...,[]


In [41]:
df.shape

(4515, 5)

In [42]:
all_labels_list = []
for labels in df.techniques:
    all_labels_list.extend(labels)
all_labels_list = list(set(all_labels_list))
all_labels_list = [elem for elem in all_labels_list if len(elem)>0]
all_labels_list

['False_Equivalence',
 'Consequential_Oversimplification',
 'Obfuscation-Vagueness-Confusion',
 'Name_Calling-Labeling',
 'Guilt_by_Association',
 'Appeal_to_Values',
 'Slogans',
 'Questioning_the_Reputation',
 'Appeal_to_Pity',
 'Exaggeration-Minimisation',
 'Whataboutism',
 'Appeal_to_Hypocrisy',
 'False_Dilemma-No_Choice',
 'Straw_Man',
 'Appeal_to_Fear-Prejudice',
 'Conversation_Killer',
 'Appeal_to_Time',
 'Appeal_to_Popularity',
 'Appeal_to_Authority',
 'Doubt',
 'Repetition',
 'Flag_Waving',
 'Causal_Oversimplification',
 'Red_Herring',
 'Loaded_Language']

In [43]:
df["persuasion_strategies"]=df.techniques.apply(lambda x: get_high_level_persuasion_group(x))

In [44]:
df

,filename,start,end,content,techniques,persuasion_strategies
0,pl_eu_12_06_2024_n02.txt,0,53,Wicemarszałek Włodzimierz Czarzasty: Dziękuję ...,[],[]
1,pl_eu_12_06_2024_n02.txt,54,143,"Witam uczniów ze Szkoły Podstawowej nr 4, szko...",[],[]
2,pl_eu_12_06_2024_n02.txt,144,256,"Witam państwa. (Oklaski) Proszę państwa, przec...",[],[]
3,pl_eu_12_06_2024_n02.txt,257,296,Ustalam czas zadania pytania na minutę.,[],[]
4,pl_eu_12_06_2024_n02.txt,297,380,Zapraszam panią poseł Annę Gembicką z Klubu Pa...,[],[]
...,...,...,...,...,...,...
4510,pl_abortion_11_04_2024_n12.txt,8197,8313,"Wynika z prawa naturalnego, jest zapisane w ko...","[Appeal_to_Authority, Appeal_to_Values]",[Justification]
4511,pl_abortion_11_04_2024_n12.txt,8314,8402,"Te przepisy, które dzisiaj reprezentujecie, są...","[Conversation_Killer, Appeal_to_Authority]","[Justification, Call]"
4512,pl_abortion_11_04_2024_n12.txt,8404,8431,Na koniec: wielki Polak św.,[Appeal_to_Authority],[Justification]
4513,pl_abortion_11_04_2024_n12.txt,8432,8561,Jan Paweł II powiedział: Jeżeli matce wolno za...,[Appeal_to_Authority],[Justification]


In [45]:
strategies = ['Attack_on_Reputation', 'Justification', 'Distraction', 'Simplification', 'Call', 'Manipulative_Wording']
df_pl_encoded = encode_persuasion_strategies(df, 'persuasion_strategies', strategies)

In [48]:
df_pl_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,pl_eu_12_06_2024_n02.txt,0,53,Wicemarszałek Włodzimierz Czarzasty: Dziękuję ...,[],[],False,False,False,False,False,False
1,pl_eu_12_06_2024_n02.txt,54,143,"Witam uczniów ze Szkoły Podstawowej nr 4, szko...",[],[],False,False,False,False,False,False
2,pl_eu_12_06_2024_n02.txt,144,256,"Witam państwa. (Oklaski) Proszę państwa, przec...",[],[],False,False,False,False,False,False
3,pl_eu_12_06_2024_n02.txt,257,296,Ustalam czas zadania pytania na minutę.,[],[],False,False,False,False,False,False
4,pl_eu_12_06_2024_n02.txt,297,380,Zapraszam panią poseł Annę Gembicką z Klubu Pa...,[],[],False,False,False,False,False,False


# Bulgarian Dataset

In [49]:
# change to your actual file path
file_path = "../data/BG/sentences.txt" # replace with your file path
df_content = read_transcript_to_dataframe(file_path, "content")
df_content.head()

,filename,start,end,content
0,20240228_URW_BG_86.txt,0,70,ЯВОР БОЖАНКОВ (ПП-ДБ): Уважаеми господин Предс...
1,20240228_URW_BG_86.txt,71,190,"Искам да Ви обясня какво направихте току-що, з..."
2,20240228_URW_BG_86.txt,192,574,Докато тя извършва агресивни действия в Украйн...
3,20240228_URW_BG_86.txt,576,668,Депутат от ГЕРБ зададе въпрос към министър Таг...
4,20240228_URW_BG_86.txt,669,793,"Изчерпателният отговор, който касае национална..."


In [50]:
file_path = "../data/BG/subtask-2-annotations-full.txt" # replace with your file path
df_labels = read_persuasion_file(file_path)

In [51]:
df_labels.head()

,filename,start,end,techniques
0,20240228_URW_BG_86.txt,0,70,[]
1,20240228_URW_BG_86.txt,71,190,[]
2,20240228_URW_BG_86.txt,192,574,[Guilt_by_Association]
3,20240228_URW_BG_86.txt,576,668,[Doubt]
4,20240228_URW_BG_86.txt,669,793,[Doubt]


In [52]:
df=df_content.merge(df_labels, on=["filename", "start", "end"])

In [53]:
df.head()

,filename,start,end,content,techniques
0,20240228_URW_BG_86.txt,0,70,ЯВОР БОЖАНКОВ (ПП-ДБ): Уважаеми господин Предс...,[]
1,20240228_URW_BG_86.txt,71,190,"Искам да Ви обясня какво направихте току-що, з...",[]
2,20240228_URW_BG_86.txt,192,574,Докато тя извършва агресивни действия в Украйн...,[Guilt_by_Association]
3,20240228_URW_BG_86.txt,576,668,Депутат от ГЕРБ зададе въпрос към министър Таг...,[Doubt]
4,20240228_URW_BG_86.txt,669,793,"Изчерпателният отговор, който касае национална...",[Doubt]


In [54]:
df.shape

(4984, 5)

In [55]:
df["persuasion_strategies"]=df.techniques.apply(lambda x: get_high_level_persuasion_group(x))

In [57]:
df.head()

,filename,start,end,content,techniques,persuasion_strategies
0,20240228_URW_BG_86.txt,0,70,ЯВОР БОЖАНКОВ (ПП-ДБ): Уважаеми господин Предс...,[],[]
1,20240228_URW_BG_86.txt,71,190,"Искам да Ви обясня какво направихте току-що, з...",[],[]
2,20240228_URW_BG_86.txt,192,574,Докато тя извършва агресивни действия в Украйн...,[Guilt_by_Association],[Attack_on_Reputation]
3,20240228_URW_BG_86.txt,576,668,Депутат от ГЕРБ зададе въпрос към министър Таг...,[Doubt],[Attack_on_Reputation]
4,20240228_URW_BG_86.txt,669,793,"Изчерпателният отговор, който касае национална...",[Doubt],[Attack_on_Reputation]


In [58]:
strategies = ['Attack_on_Reputation', 'Justification', 'Distraction', 'Simplification', 'Call', 'Manipulative_Wording']
df_bg_encoded = encode_persuasion_strategies(df, 'persuasion_strategies', strategies)

In [59]:
df_bg_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,20240228_URW_BG_86.txt,0,70,ЯВОР БОЖАНКОВ (ПП-ДБ): Уважаеми господин Предс...,[],[],False,False,False,False,False,False
1,20240228_URW_BG_86.txt,71,190,"Искам да Ви обясня какво направихте току-що, з...",[],[],False,False,False,False,False,False
2,20240228_URW_BG_86.txt,192,574,Докато тя извършва агресивни действия в Украйн...,[Guilt_by_Association],[Attack_on_Reputation],True,False,False,False,False,False
3,20240228_URW_BG_86.txt,576,668,Депутат от ГЕРБ зададе въпрос към министър Таг...,[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
4,20240228_URW_BG_86.txt,669,793,"Изчерпателният отговор, който касае национална...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False


In [91]:
df_bg_encoded.shape

(4984, 12)

# Russian Dataset

In [60]:
# change to your actual file path
file_path = "../data/RU/sentences.txt" # replace with your file path
df_content = read_transcript_to_dataframe(file_path, "content")
df_content.head()

,filename,start,end,content
0,RU_URW_14.txt,0,231,США прямо участвовали в убийстве российских со...
1,RU_URW_14.txt,235,460,"Как написали коллеги, все маломальские успехи ..."
2,RU_URW_14.txt,464,633,"Украинская армия не играла никакой роли, а ее ..."
3,RU_URW_14.txt,637,768,США хотели знать - будет ли разведка и высокот...
4,RU_URW_14.txt,769,914,"Будут ли украинские дроноводы, ракетчики, пило..."


In [61]:
file_path = "../data/RU/subtask-2-annotations-full.txt" # replace with your file path
df_labels = read_persuasion_file(file_path)

In [62]:
df_labels.head()

,filename,start,end,techniques
0,RU_URW_14.txt,0,231,[Causal_Oversimplification]
1,RU_URW_14.txt,235,460,[Doubt]
2,RU_URW_14.txt,464,633,[Doubt]
3,RU_URW_14.txt,637,768,[Repetition]
4,RU_URW_14.txt,769,914,[Repetition]


In [63]:
df=df_content.merge(df_labels, on=["filename", "start", "end"])

In [64]:
df.head()

,filename,start,end,content,techniques
0,RU_URW_14.txt,0,231,США прямо участвовали в убийстве российских со...,[Causal_Oversimplification]
1,RU_URW_14.txt,235,460,"Как написали коллеги, все маломальские успехи ...",[Doubt]
2,RU_URW_14.txt,464,633,"Украинская армия не играла никакой роли, а ее ...",[Doubt]
3,RU_URW_14.txt,637,768,США хотели знать - будет ли разведка и высокот...,[Repetition]
4,RU_URW_14.txt,769,914,"Будут ли украинские дроноводы, ракетчики, пило...",[Repetition]


In [65]:
df.shape

(2074, 5)

In [66]:
df["persuasion_strategies"]=df.techniques.apply(lambda x: get_high_level_persuasion_group(x))

In [67]:
df.head()

,filename,start,end,content,techniques,persuasion_strategies
0,RU_URW_14.txt,0,231,США прямо участвовали в убийстве российских со...,[Causal_Oversimplification],[Simplification]
1,RU_URW_14.txt,235,460,"Как написали коллеги, все маломальские успехи ...",[Doubt],[Attack_on_Reputation]
2,RU_URW_14.txt,464,633,"Украинская армия не играла никакой роли, а ее ...",[Doubt],[Attack_on_Reputation]
3,RU_URW_14.txt,637,768,США хотели знать - будет ли разведка и высокот...,[Repetition],[Manipulative_Wording]
4,RU_URW_14.txt,769,914,"Будут ли украинские дроноводы, ракетчики, пило...",[Repetition],[Manipulative_Wording]


In [68]:
strategies = ['Attack_on_Reputation', 'Justification', 'Distraction', 'Simplification', 'Call', 'Manipulative_Wording']
df_ru_encoded = encode_persuasion_strategies(df, 'persuasion_strategies', strategies)

In [69]:
df_ru_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,RU_URW_14.txt,0,231,США прямо участвовали в убийстве российских со...,[Causal_Oversimplification],[Simplification],False,False,False,True,False,False
1,RU_URW_14.txt,235,460,"Как написали коллеги, все маломальские успехи ...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
2,RU_URW_14.txt,464,633,"Украинская армия не играла никакой роли, а ее ...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
3,RU_URW_14.txt,637,768,США хотели знать - будет ли разведка и высокот...,[Repetition],[Manipulative_Wording],False,False,False,False,False,True
4,RU_URW_14.txt,769,914,"Будут ли украинские дроноводы, ракетчики, пило...",[Repetition],[Manipulative_Wording],False,False,False,False,False,True


# Train/Valid/Test Split

In [70]:
df_pl_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,pl_eu_12_06_2024_n02.txt,0,53,Wicemarszałek Włodzimierz Czarzasty: Dziękuję ...,[],[],False,False,False,False,False,False
1,pl_eu_12_06_2024_n02.txt,54,143,"Witam uczniów ze Szkoły Podstawowej nr 4, szko...",[],[],False,False,False,False,False,False
2,pl_eu_12_06_2024_n02.txt,144,256,"Witam państwa. (Oklaski) Proszę państwa, przec...",[],[],False,False,False,False,False,False
3,pl_eu_12_06_2024_n02.txt,257,296,Ustalam czas zadania pytania na minutę.,[],[],False,False,False,False,False,False
4,pl_eu_12_06_2024_n02.txt,297,380,Zapraszam panią poseł Annę Gembicką z Klubu Pa...,[],[],False,False,False,False,False,False


In [71]:
df_ru_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,RU_URW_14.txt,0,231,США прямо участвовали в убийстве российских со...,[Causal_Oversimplification],[Simplification],False,False,False,True,False,False
1,RU_URW_14.txt,235,460,"Как написали коллеги, все маломальские успехи ...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
2,RU_URW_14.txt,464,633,"Украинская армия не играла никакой роли, а ее ...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
3,RU_URW_14.txt,637,768,США хотели знать - будет ли разведка и высокот...,[Repetition],[Manipulative_Wording],False,False,False,False,False,True
4,RU_URW_14.txt,769,914,"Будут ли украинские дроноводы, ракетчики, пило...",[Repetition],[Manipulative_Wording],False,False,False,False,False,True


In [72]:
df_bg_encoded.head()

,filename,start,end,content,techniques,persuasion_strategies,Attack_on_Reputation,Justification,Distraction,Simplification,Call,Manipulative_Wording
0,20240228_URW_BG_86.txt,0,70,ЯВОР БОЖАНКОВ (ПП-ДБ): Уважаеми господин Предс...,[],[],False,False,False,False,False,False
1,20240228_URW_BG_86.txt,71,190,"Искам да Ви обясня какво направихте току-що, з...",[],[],False,False,False,False,False,False
2,20240228_URW_BG_86.txt,192,574,Докато тя извършва агресивни действия в Украйн...,[Guilt_by_Association],[Attack_on_Reputation],True,False,False,False,False,False
3,20240228_URW_BG_86.txt,576,668,Депутат от ГЕРБ зададе въпрос към министър Таг...,[Doubt],[Attack_on_Reputation],True,False,False,False,False,False
4,20240228_URW_BG_86.txt,669,793,"Изчерпателният отговор, който касае национална...",[Doubt],[Attack_on_Reputation],True,False,False,False,False,False


In [81]:
# Step 1: Sample 500 examples
test_ru = df_ru_encoded.sample(n=500, random_state=42)
# Step 2: Remove these 500 from the original DataFrame
remaining_ru = df_ru_encoded.drop(test_ru.index)
# Step 3: Split remaining data into train (70%) and validation (30%)
train_ru, val_ru = train_test_split(remaining_ru, test_size=0.3, random_state=42)

In [82]:
# Check sizes
print(f"Test set: {len(test_ru)}")
print(f"Train set: {len(train_ru)}")
print(f"Validation set: {len(val_ru)}")

Test set: 500
Train set: 1101
Validation set: 473


In [88]:
test_ru.to_csv("../data/RU/test.csv", index=False)
train_ru.to_csv("../data/RU/train.csv", index=False)
val_ru.to_csv("../data/RU/valid.csv", index=False)

In [83]:
# Step 1: Sample 500 examples
test_pl = df_pl_encoded.sample(n=500, random_state=42)
# Step 2: Remove these 500 from the original DataFrame
remaining_pl = df_pl_encoded.drop(test_ru.index)
# Step 3: Split remaining data into train (70%) and validation (30%)
train_pl, val_pl = train_test_split(remaining_pl, test_size=0.3, random_state=42)

In [84]:
# Check sizes
print(f"Test set: {len(test_pl)}")
print(f"Train set: {len(train_pl)}")
print(f"Validation set: {len(val_pl)}")

Test set: 500
Train set: 2810
Validation set: 1205


In [89]:
test_pl.to_csv("../data/PL/test.csv", index=False)
train_pl.to_csv("../data/PL/train.csv", index=False)
val_pl.to_csv("../data/PL/valid.csv", index=False)

In [85]:
# Step 1: Sample 500 examples
test_bg = df_bg_encoded.sample(n=500, random_state=42)
# Step 2: Remove these 500 from the original DataFrame
remaining_bg = df_bg_encoded.drop(test_bg.index)
# Step 3: Split remaining data into train (70%) and validation (30%)
train_bg, val_bg = train_test_split(remaining_bg, test_size=0.3, random_state=42)

In [86]:
# Check sizes
print(f"Test set: {len(test_bg)}")
print(f"Train set: {len(train_bg)}")
print(f"Validation set: {len(val_bg)}")

Test set: 500
Train set: 3138
Validation set: 1346


In [90]:
test_bg.to_csv("../data/BG/test.csv", index=False)
train_bg.to_csv("../data/BG/train.csv", index=False)
val_bg.to_csv("../data/BG/valid.csv", index=False)